# ISIC 2020 Classification Training (Ultralytics YOLO)

This notebook trains a custom skin-lesion classifier using the ISIC 2020 dataset (binary melanoma).

**You will need to download the dataset manually** (license + account required) or via Kaggle.
Set the paths below to point at your local dataset folder before running the prep cell.


## 1. Install Dependencies

Run once per environment. If you already installed the repo requirements, you can skip this cell.


In [ ]:
import sys
!{sys.executable} -m pip install -r ../requirements.txt

## 2. Dataset Setup

Download ISIC 2020 data and place it locally. Expected layout:

- `train.csv`
- `jpeg/train/` (images)

Set `RAW_DATA_DIR` to the folder containing those files.


In [ ]:
from pathlib import Path
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

RAW_DATA_DIR = Path("../data/isic2020_raw")
OUTPUT_DIR = Path("../data/isic2020_yolo")
SPLIT_SEED = 42
VAL_FRACTION = 0.1
TEST_FRACTION = 0.1
MAX_SAMPLES = None  # e.g., 200 for a quick demo


def safe_link_or_copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def load_isic_2020():
    images_dir = RAW_DATA_DIR / "jpeg" / "train"
    labels_csv = RAW_DATA_DIR / "train.csv"
    if not labels_csv.exists():
        raise FileNotFoundError(f"Missing CSV: {labels_csv}")
    df = pd.read_csv(labels_csv)
    df["label"] = df["target"].map({0: "benign", 1: "melanoma"})
    df["filename"] = df["image_name"] + ".jpg"
    return images_dir, df[["filename", "label"]]


def sample_per_class(df, max_samples):
    if max_samples is None:
        return df
    return (
        df.groupby("label", group_keys=False)
        .apply(lambda group: group.sample(min(len(group), max_samples), random_state=SPLIT_SEED))
        .reset_index(drop=True)
    )


images_dir, df = load_isic_2020()

df = sample_per_class(df, MAX_SAMPLES)
train_df, temp_df = train_test_split(
    df, test_size=VAL_FRACTION + TEST_FRACTION, random_state=SPLIT_SEED, stratify=df["label"]
)
val_size = TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION)
val_df, test_df = train_test_split(
    temp_df, test_size=val_size, random_state=SPLIT_SEED, stratify=temp_df["label"]
)


def write_split(split_df, split_name):
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"{split_name} split"):
        src = images_dir / row["filename"]
        if not src.exists():
            raise FileNotFoundError(f"Missing image: {src}")
        dst = OUTPUT_DIR / split_name / row["label"] / row["filename"]
        safe_link_or_copy(src, dst)


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_split(train_df, "train")
write_split(val_df, "val")
write_split(test_df, "test")

print("Dataset prepared at", OUTPUT_DIR.resolve())


## 3. Train the Classifier

If you do not have a GPU, set `device='cpu'` and reduce the batch size.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n-cls.pt")
results = model.train(
    data=str(OUTPUT_DIR),
    epochs=20,
    imgsz=224,
    batch=32,
    device=0,
    project="runs",
    name="isic2020_cls"
)


## 4. Evaluate on the Test Split + Quick Inference

Use the test split created in the prep step. For a CLI-only workflow, you can also run:

```bash
python scripts/classify/test_isic2020.py --model runs/isic2020_cls/weights/best.pt --data data/isic2020_yolo
```


In [ ]:
metrics = model.val(data=str(OUTPUT_DIR), split="test")
print(metrics)

sample_image = next((OUTPUT_DIR / "val").rglob("*.jpg"))
preds = model.predict(source=str(sample_image), save=True)
preds[0].show()
